In [ ]:
!nvidia-smi
!pip install ultralytics roboflow -q

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="COLOQUE_SUA_CHAVE_AQUI")

project = rf.workspace("antonio-raimundo").project("crack-detection-y5kyg")
dataset = project.version(2).download("yolov8")

print("Dataset salvo em:", dataset.location)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    name="fissuras_encosta",
)

In [ ]:
metrics = model.val()
print(f"mAP50:    {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")

In [ ]:
import glob

test_images = glob.glob(f"{dataset.location}/test/images/*.jpg")[:3]
results = model.predict(test_images, save=True, conf=0.25)

for r in results:
    r.show()

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import shutil
shutil.copy(
    "runs/detect/fissuras_encosta/weights/best.pt",
    "/content/drive/MyDrive/best_yolov8_rachadura.pt",
)
print("Modelo salvo no Drive.")